<a href="https://colab.research.google.com/github/nova0816/Agentic_agent/blob/main/AgenticAI_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install the necessary library
!pip install google-genai

In [2]:
# --- 1. Local Data/Database Simulation ---
# Our simple, local weather table
WEATHER_DATA = {
    "2025-12-16": {"city": "Taipei", "temp": "18°C", "condition": "Cloudy"},
    "2025-12-17": {"city": "Taipei", "temp": "22°C", "condition": "Sunny"},
    "2025-12-18": {"city": "Taipei", "temp": "15°C", "condition": "Rainy"},
    "2025-12-19": {"city": "Taipei", "temp": "20°C", "condition": "Partly Cloudy"},
}


# --- 2. The Python Function (Tool) ---
def get_weather(date: str) -> str:
    """
    Returns the weather for a specific date from the local database.
    The date must be in YYYY-MM-DD format.
    """
    if date in WEATHER_DATA:
        data = WEATHER_DATA[date]
        return f"On {date}, the weather is {data['condition']} with a temperature of {data['temp']} in {data['city']}."
    else:
        return f"Sorry, I do not have weather data for {date} in my local database."


# --- 3. The Core Agent Logic ---

import os
import google.generativeai as genai
from google.colab import userdata

# Define the model name as a string constant
MODEL_NAME_STR = "gemini-2.5-flash"

# API Key Setup
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    os.environ['GEMINI_API_KEY'] = API_KEY
except:
    print("Warning: 'GEMINI_API_KEY' not found in Colab Secrets. Please create it or set the API_KEY variable manually.")
    # Fallback for manual entry
    if 'API_KEY' not in locals():
        API_KEY = input("Enter your Gemini API Key: ")
        os.environ['GEMINI_API_KEY'] = API_KEY

# Explicitly configure the generative AI with the API key
genai.configure(api_key=API_KEY)

# A list of tools the model can use. This will be passed to the API.
tools_list = [get_weather]

# Initialize the GenerativeModel with tools, using the string constant for the model name
generative_model = genai.GenerativeModel(model_name=MODEL_NAME_STR, tools=tools_list)

def run_agent(prompt: str):
    """
    Runs the prompt through the LLM, handles function calls,
    and returns the final response.
    """
    print(f"User Prompt: '{prompt}'")
    print("-" * 30)

    # 1. First Call: Send the prompt and tools to the LLM
    response = generative_model.generate_content(
        contents=[prompt]
    )

    # Check if the LLM decided to call a function
    function_call_part = None
    try:
        function_call_part = response.candidates[0].content.parts[0].function_call
    except (AttributeError, IndexError):
        pass # No function call in the response

    if not function_call_part:
        print("LLM Response (No Function Call):")
        return response.text

    # --- 4. Execute the Function in Python ---

    # Assuming only one function call for simplicity
    call = function_call_part
    function_name = call.name
    function_args = dict(call.args)

    print(f"✅ LLM requested function call:")
    print(f"   Function: {function_name}")
    print(f"   Arguments: {function_args}")

    # Safely look up and execute the function
    if function_name == "get_weather":
        # Call the actual Python function with the arguments provided by the LLM
        function_result = get_weather(**function_args)
    else:
        function_result = "Error: Unknown function requested by LLM."

    print(f"\n💻 Function execution result: '{function_result}'")

    # --- 5. Second Call: Send Result Back to the LLM ---
    # Construct the function response part as a dictionary
    function_response_part = {
        "function_response": {
            "name": function_name,
            "response": {"result": function_result}
        }
    }

    # Send the original prompt, the LLM's function call, and the function's output
    final_response = generative_model.generate_content(
        contents=[prompt, function_call_part, function_response_part]
    )

    print("\n📝 Final LLM Response to User:")
    return final_response.text

In [4]:
# --- Test Case 1: Function Call Triggered ---
# The LLM sees the word "weather" and the date, and knows to use the 'get_weather' tool.
print("--- TEST 1: Should trigger get_weather ---")
result_1 = run_agent("What is the weather in the local database for 2025-12-17?")
print(result_1)
print("\n" * 2)

--- TEST 1: Should trigger get_weather ---
User Prompt: 'What is the weather in the local database for 2025-12-17?'
------------------------------
✅ LLM requested function call:
   Function: get_weather
   Arguments: {'date': '2025-12-17'}

💻 Function execution result: 'On 2025-12-17, the weather is Sunny with a temperature of 22°C in Taipei.'

📝 Final LLM Response to User:
On 2025-12-17, the weather is Sunny with a temperature of 22°C in Taipei.





In [ ]:
# --- Test Case 2: Function Call Triggered (Different Date) ---
print("--- TEST 2: Should trigger get_weather and return 'Rainy' ---")
result_2 = run_agent("I need to know the forecast for 2025-12-18, what will the weather be?")
print(result_2)
print("\n" * 2)

# --- Test Case 3: No Function Call ---
# The LLM recognizes that this is a general question and does not need the weather tool.
print("--- TEST 3: Should NOT trigger a function ---")
result_3 = run_agent("What is the capital of France?")
print(result_3)

In [7]:
response = generative_model.generate_content(contents=["how is the weather in Dec.18, 2025"])

In [8]:
response

response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "candidates": [
        {
          "content": {
            "parts": [
              {
                "function_call": {
                  "name": "get_weather",
                  "args": {
                    "date": "2025-12-18"
                  }
                }
              }
            ],
            "role": "model"
          },
          "finish_reason": "STOP",
          "index": 0
        }
      ],
      "usage_metadata": {
        "prompt_token_count": 80,
        "candidates_token_count": 24,
        "total_token_count": 228
      },
      "model_version": "gemini-2.5-flash"
    }),
)